# 05 – Seleção de Features

A base pós-integração continha 665 colunas. Modelos preditivos com dimensionalidade excessiva estão sujeitos a overfitting, multicolinearidade e perda de interpretabilidade clínica. Para mitigar esses riscos, foi aplicado um processo estruturado de redução de dimensionalidade em cinco etapas sequenciais:

1. **Remoção conceitual:** Variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Ex: `motivo_saida`.
2. **Variáveis constantes:** Colunas nas quais todos os registros têm o mesmo valor.
3. **Variáveis quasi-constantes:** Colunas nas quais mais de 98% dos registros assumem o mesmo valor.
4. **Baixa correlação com o alvo:** Correlação de Pearson com `indicador_obito` inferior a |r| = 0,03.
5. **Alta ausência:** Proporção de ausência superior a 99% e textos redundantes.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Paths
ROOT = Path("..").resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'base_modelagem.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'base_modelagem_reduzida.csv'

# Config
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# Carregamento da base
df = pd.read_csv(DATA_PATH, low_memory=False)

# Garantir que indicador_obito seja numérico
df['indicador_obito'] = pd.to_numeric(df['indicador_obito'], errors='coerce')

print(f"Formato inicial: {df.shape[0]} linhas e {df.shape[1]} colunas.")


Formato inicial: 415367 linhas e 658 colunas.


In [3]:
for i, col in enumerate(df.columns, 1):
    print(
        i,
        col,
        df[col].dtype,
        df[col].dropna().iloc[0] if df[col].notna().any() else "Sem valores"
    )

1 ano_competencia int64 2015
2 mes_competencia int64 1
3 especialidade_leito_cod int64 3
4 cnpj_hospital float64 57740490000260.0
5 numero_aih_cod int64 3514116185327
6 cep_paciente int64 11713110
7 municipio_residencia_cod int64 354100
8 data_nascimento object 1957-11-24
9 uti_mes_inicial int64 0
10 uti_mes_anterior int64 0
11 uti_mes_alta int64 0
12 uti_mes_total_cod int64 0
13 uti_intermediaria_inicial int64 0
14 uti_intermediaria_anterior int64 0
15 uti_intermediaria_alta int64 0
16 uti_intermediaria_total int64 0
17 diarias_acompanhante int64 0
18 quantidade_diarias int64 28
19 procedimento_solicitado int64 303060190
20 procedimento_realizado_cod int64 303060190
21 valor_servicos_hospitalares float64 748.75
22 valor_servicos_profissionais float64 170.9
23 valor_sadt float64 0.0
24 valor_rn float64 0.0
25 valor_acompanhante float64 0.0
26 valor_ortese_protese float64 0.0
27 valor_sangue float64 0.0
28 valor_sadtsr float64 0.0
29 valor_transporte float64 0.0
30 valor_obsang float64 

## Etapa 1: Remoção por critério conceitual
Removendo variáveis com vazamento de dados (data leakage), identificadores únicos sem poder preditivo e variáveis redundantes. Destaque para `motivo_saida`, cujo preenchimento equivale a antecipar o próprio desfecho.

In [4]:
# Colunas que representam leakage, identificadores ou data release posterior ao evento
leakage_keywords = [
        'motivo_saida', 'data_saida', 'cid_morte', 'numero_aih',
    'numero_remessa', 'sequencial', 'cnpj_hospital', 'cep_paciente',
    'cpf_gestor', 'cnpj_mantenedora', 'sequencial_remessa', 'cep_estabelecimento',
    'cpf_cnpj_estabelecimento', 'cnes_cnpj_mantenedora', 'arquivo_origem',
    'data_internacao', 'ap01cv07', 'ap02cv07', 'ap03cv07', 'ap04cv07', 'ap05cv07', 'ap06cv07', 'ap07cv07', 'dt_atual',
    # --- Adicionadas para evitar vazamento e ruído geográfico/financeiro ---
    'valor_servicos_hospitalares', 'valor_servicos_profissionais', 'valor_total', 'valor_uti', 'valor_total_dolar', 'custo',
    'diarias_acompanhante', 'quantidade_diarias', 'diarias', 'dias_permanencia_cod', 'uti_mes_total_cod', 'tipo_uti',
    'municipio_residencia_cod', 'municipio_estabelecimento_cod', 'codigo_municipio', 'regiao_saude', 'distrito_sanitario', 'codigo_banco',
    'lavanderia', 'necroterio', 'lactario', 'procedimento_solicitado', 'mes_competencia', 'competencia_cod',
    'regra_contratual', 'centro_obstetrico', 'unidade_neonatal', 'banco_leite', 'orgao_expedidor',
    'comissao_', 'coleta_residuo', 'servico_apoio', 'gestao_ab', 'gestao_mc', 'gestao_ac', 'gestao_programa',
    'convenio_particular', 'plano_publico', 'plano_privado', 'convenio_sus', 'fluxo_clientela', 'tipo_prestador'
]

cols_to_drop_step1 = []
for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in leakage_keywords):
        cols_to_drop_step1.append(col)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Algumas colunas que devem ser removidas por match exato
exact_drop = ['competencia']
for c in exact_drop:
    if c in df.columns:
        cols_to_drop_step1.append(c)

cols_to_drop_step1 = list(set(cols_to_drop_step1))
# Garantir que indicador_obito nunca seja removido acidentalmente
if 'indicador_obito' in cols_to_drop_step1:
    cols_to_drop_step1.remove('indicador_obito')

df.drop(columns=[c for c in cols_to_drop_step1 if c in df.columns], inplace=True)
print(f"Removidas {len(cols_to_drop_step1)} colunas na Etapa 1.")
print(f"Formato atual: {df.shape}")


Removidas 128 colunas na Etapa 1.
Formato atual: (415367, 530)


## Etapa 2: Remoção de variáveis constantes
Eliminação de variáveis numéricas constantes — aquelas com valor único em toda a base. Essas não possuem capacidade discriminativa.

In [5]:
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
df.drop(columns=constantes, inplace=True)
print(f"Removidas {len(constantes)} colunas constantes na Etapa 2.")
print(f"Exemplos apagados: {constantes[:10]}")
print(f"Formato atual: {df.shape}")


Removidas 54 colunas constantes na Etapa 2.
Exemplos apagados: ['uti_mes_inicial', 'uti_mes_anterior', 'uti_mes_alta', 'uti_intermediaria_inicial', 'uti_intermediaria_anterior', 'uti_intermediaria_alta', 'uti_intermediaria_total', 'valor_sadt', 'valor_rn', 'valor_acompanhante']
Formato atual: (415367, 476)


## Etapa 3: Remoção de variáveis quasi-constantes
Eliminação de variáveis quasi-constantes, definidas como colunas nas quais mais de 98% dos registros assumem o mesmo valor.

In [6]:
quasi_constantes = []
for c in df.columns:
    if c != 'indicador_obito':
        top_freq = df[c].value_counts(normalize=True, dropna=False).iloc[0]
        if top_freq > 0.98:
            quasi_constantes.append(c)

df.drop(columns=quasi_constantes, inplace=True)
print(f"Removidas {len(quasi_constantes)} colunas quasi-constantes (>98%) na Etapa 3.")
print(f"Exemplos apagados: {quasi_constantes[:10]}")
print(f"Formato atual: {df.shape}")


Removidas 117 colunas quasi-constantes (>98%) na Etapa 3.
Exemplos apagados: ['codigo_idade_cod', 'nacionalidade_cod', 'indicador_homonimo', 'valor_sh_federal', 'valor_sp_federal', 'tipo_diag_sec_5_cod', 'tipo_diag_sec_6_cod', 'tipo_diag_sec_7_cod', 'tipo_diag_sec_8_cod', 'tipo_diag_sec_9_cod']
Formato atual: (415367, 359)


## Etapa 4: Correlação com o alvo — calculada apenas no treino (NB06)

O filtro de correlação de Pearson com `indicador_obito` **não é aplicado aqui** para evitar vazamento de dados (*data leakage*): calculá-lo no dataset completo deixaria informação do conjunto de teste influenciar a seleção de variáveis.

Esta célula apenas **registra o critério** (|r| ≥ 0,03). O filtro é executado no `06_predictive_modeling.ipynb`, exclusivamente sobre o conjunto de treino (2015–2022), e o resultado é aplicado via `.reindex()` ao conjunto de teste.

In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'indicador_obito' in num_cols:
    num_cols.remove('indicador_obito')

print(f'Colunas numéricas disponíveis para filtro de correlação: {len(num_cols)}')
print('Filtro |r| >= 0.03 será aplicado no NB06 após o split temporal.')
print(f'Formato atual (sem alteração): {df.shape}')


Colunas numéricas disponíveis para filtro de correlação: 337
Filtro |r| >= 0.03 será aplicado no NB06 após o split temporal.
Formato atual (sem alteração): (415367, 359)


## Etapa 5: Alta ausência e textos redundantes
Exclusão de variáveis com proporção de ausência superior a 99% e variáveis textuais redundantes/não codificadas.

In [8]:
alta_ausencia = [c for c in df.columns if df[c].isna().mean() > 0.99]
df.drop(columns=alta_ausencia, inplace=True)
print(f"Removidas {len(alta_ausencia)} colunas com ausência > 99% na Etapa 5.")
print(f"Exemplos apagados: {alta_ausencia[:10]}")

# Textos redundantes ou com cardinalidade extrema (exceto codigo_cnes, que pode ser agrupado futuramente se desejado, mas aqui será limpo caso necessário).
# Para modelagem, vamos dropar objects não convertidos
text_cols = df.select_dtypes(include=['object']).columns.tolist()
cols_text_drop = [c for c in text_cols if df[c].nunique() > 100]
if 'codigo_cnes' in cols_text_drop:
    cols_text_drop.remove('codigo_cnes') # Manter cnes para referencial, se precisar

df.drop(columns=cols_text_drop, inplace=True)
print(f"Removidas {len(cols_text_drop)} colunas de texto puras/alta cardinalidade.")
print(f"Apagadas: {cols_text_drop}")

print(f"Formato atual: {df.shape}")


Removidas 0 colunas com ausência > 99% na Etapa 5.
Exemplos apagados: []
Removidas 7 colunas de texto puras/alta cardinalidade.
Apagadas: ['data_nascimento', 'diagnostico_secundario_1', 'municipio_gestor', 'codigo_agencia', 'conta_corrente', 'numero_alvara', 'data_expedicao_alvara_cod']
Formato atual: (415367, 352)


## Etapa 6: Tratamento de Categorias de Alta Cardinalidade
Variáveis terminadas em `_cod`, além de `tipo_gestor` e `procedimento_solicitado`, são códigos numéricos que representam categorias. Se deixadas como números, os algoritmos (e o SHAP) assumirão que há uma relação de ordem entre elas, o que é estatisticamente incorreto.

Vamos transformá-las em texto para que sejam consideradas como variáveis dummy no próximo notebook. Para evitar que centenas de novas colunas sejam criadas, agruparemos os códigos raros na categoria `'Outros'`, preservando apenas o Top 10 de cada coluna.

Obs: as variáveis binárias (`0/1`) provenientes de CNES (como `habilitacao_`, `equip_`, `servico_`) já são numéricas e prontas para o modelo, portanto não passam por esta etapa.

In [9]:
cod_cols = [c for c in df.columns if c.endswith('_cod') or c in ['procedimento_solicitado', 'tipo_gestor']]

print(f"Tratando {len(cod_cols)} colunas categóricas...")
print(f"Variáveis afetadas: {cod_cols}\n")

for c in cod_cols:
    if c in df.columns:
        df[c] = df[c].astype(str) # Força a ser string (categoria)
        
        # Mantém as 10 mais frequentes e joga o resto para 'Outros'
        top_cats = df[c].value_counts().nlargest(10).index
        df[c] = df[c].where(df[c].isin(top_cats), 'Outros')

# Algumas outras colunas categóricas textuais devem ser garantidas como string para o get_dummies:
outras_cats = ['sexo', 'carater_internacao', 'raca_cor']
for c in outras_cats:
    if c in df.columns:
        df[c] = df[c].astype(str)

print("Tratamento concluído.")

Tratando 8 colunas categóricas...
Variáveis afetadas: ['especialidade_leito_cod', 'procedimento_realizado_cod', 'diagnostico_principal_cod', 'tipo_gestor', 'complexidade_cod', 'tipo_diag_sec_2_cod', 'tipo_diag_sec_3_cod', 'tipo_diag_sec_4_cod']

Tratamento concluído.


## Etapa 7: Revisão Final dos Grupos de Variáveis (CNES e Clínicas)

Nesta etapa, vamos mapear o que sobrou após todos os cortes automáticos. Como você bem observou, variáveis como `habilitacao_` já são *dummies* naturais (0 = não tem, 1 = tem) e já estão prontas para o modelo.

Abaixo nós imprimimos um resumo de quantas colunas restaram por categoria. Caso você queira excluir grupos inteiros que não façam sentido clínico para Infarto (por exemplo, serviços ambulatoriais ou habilitações que não sejam de cardiologia/UTI), você pode listá-los em `colunas_para_remover`.

In [10]:
todas = df.columns.tolist()
habs = [c for c in todas if c.startswith('habilitacao_')]
equips = [c for c in todas if c.startswith('equip_')]
servs = [c for c in todas if c.startswith('servico_')]
cods = [c for c in todas if c.endswith('_cod') or c in ['procedimento_solicitado', 'tipo_gestor']]
outras = [c for c in todas if c not in habs + equips + servs + cods]

print(f"--- RESUMO DAS {len(todas)} COLUNAS RESTANTES ---")
print(f"🏥 Habilitações CNES (dummies naturais 0/1): {len(habs)}")
print(f"🛠 Equipamentos CNES: {len(equips)}")
print(f"🩺 Serviços CNES: {len(servs)}")
print(f"🔢 Códigos (Top 10 + 'Outros'): {len(cods)}")
print(f"👤 Outras (Clínicas, Demográficas, Desfecho): {len(outras)}\n")

# Exemplo: Se você quiser ver o nome de todas as habilitações que sobraram, descomente a linha abaixo:
# print("Habilitações restantes:", habs)

# ==================================================================
# REMOÇÃO MANUAL OPCIONAL
# Coloque aqui o nome de colunas específicas que você olhou e viu que não fazem 
# sentido preditivo para Infarto Agudo do Miocárdio (ex: município).
# ==================================================================
colunas_para_remover = [
    # 'municipio_residencia_cod', 
    # 'municipio_estabelecimento_cod'
]

if colunas_para_remover:
    df.drop(columns=[c for c in colunas_para_remover if c in df.columns], inplace=True)
    print(f"Removidas {len(colunas_para_remover)} colunas manualmente.")
    print(f"Formato atual: {df.shape}")

--- RESUMO DAS 352 COLUNAS RESTANTES ---
🏥 Habilitações CNES (dummies naturais 0/1): 123
🛠 Equipamentos CNES: 86
🩺 Serviços CNES: 54
🔢 Códigos (Top 10 + 'Outros'): 8
👤 Outras (Clínicas, Demográficas, Desfecho): 81



In [11]:
todas = df.columns.tolist()
todas

for coluna in df.columns:
    print(coluna, "->", df[coluna].dtype)

ano_competencia -> int64
especialidade_leito_cod -> object
procedimento_realizado_cod -> object
diagnostico_principal_cod -> object
idade -> int64
indicador_obito -> int64
motivo_autorizacao -> int64
tipo_gestor -> object
codigo_cnes -> int64
complexidade_cod -> object
tipo_diag_sec_2_cod -> object
tipo_diag_sec_3_cod -> object
tipo_diag_sec_4_cod -> object
sexo -> object
natureza_hospital -> object
natureza_juridica -> object
tipo_gestao -> object
carater_internacao -> object
tipo_financiamento -> object
raca_cor -> object
tipo_diag_sec_1 -> object
idade_num -> int64
nivel_atencao_ambulatorial -> int64
qtd_leitos_cirurgicos -> int64
qtd_leitos_clinicos -> int64
qtd_leitos_complementares -> int64
qtd_instalacao_01 -> int64
qtd_instalacao_02 -> int64
qtd_instalacao_03 -> int64
qtd_instalacao_04 -> int64
qtd_instalacao_05 -> int64
qtd_instalacao_06 -> int64
qtd_instalacao_07 -> int64
qtd_instalacao_08 -> int64
qtd_instalacao_09 -> int64
qtd_instalacao_10 -> int64
qtd_instalacao_11 -> int

In [12]:
# 1. Pegar as listas de colunas
habs = [c for c in df.columns if c.startswith('habilitacao_')]
equips = [c for c in df.columns if c.startswith('equip_')]
servs = [c for c in df.columns if c.startswith('servico_')]
qtds = [c for c in df.columns if c.startswith('qtd_')]
# 2. Preencher possíveis vazios com 0 (se estava vazio, o hospital não tinha o recurso)
# e converter para formatos inteiros muito mais leves
df[habs + equips + servs] = df[habs + equips + servs].fillna(0).astype('int8')
df[qtds] = df[qtds].fillna(0).astype('int32')
print("Tipos otimizados para economizar memória!")

Tipos otimizados para economizar memória!


In [13]:
for coluna in df.columns:
    print(coluna, "->", df[coluna].dtype)

ano_competencia -> int64
especialidade_leito_cod -> object
procedimento_realizado_cod -> object
diagnostico_principal_cod -> object
idade -> int64
indicador_obito -> int64
motivo_autorizacao -> int64
tipo_gestor -> object
codigo_cnes -> int64
complexidade_cod -> object
tipo_diag_sec_2_cod -> object
tipo_diag_sec_3_cod -> object
tipo_diag_sec_4_cod -> object
sexo -> object
natureza_hospital -> object
natureza_juridica -> object
tipo_gestao -> object
carater_internacao -> object
tipo_financiamento -> object
raca_cor -> object
tipo_diag_sec_1 -> object
idade_num -> int64
nivel_atencao_ambulatorial -> int64
qtd_leitos_cirurgicos -> int32
qtd_leitos_clinicos -> int32
qtd_leitos_complementares -> int32
qtd_instalacao_01 -> int32
qtd_instalacao_02 -> int32
qtd_instalacao_03 -> int32
qtd_instalacao_04 -> int32
qtd_instalacao_05 -> int32
qtd_instalacao_06 -> int32
qtd_instalacao_07 -> int32
qtd_instalacao_08 -> int32
qtd_instalacao_09 -> int32
qtd_instalacao_10 -> int32
qtd_instalacao_11 -> int

## Salvamento da Base Reduzida

In [15]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Base reduzida salva em: {PROCESSED_PATH}")
print(f"Formato final: {df.shape[0]} linhas e {df.shape[1]} colunas.")

Base reduzida salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem_reduzida.csv
Formato final: 415367 linhas e 352 colunas.
